In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# -----------------------
# 1) metric.csv summaries
# -----------------------
metric_cmdbs = pd.Series(metric['cmdb_id'].unique(), name='cmdb_id')
metric_cmdbs_count = metric['cmdb_id'].nunique()

metric_kpis = pd.Series(metric['kpi_name'].unique()[:50], name='kpi_name')  # limit to 50 names
metric_kpis_count = metric['kpi_name'].nunique()

# For each (cmdb_id, kpi_name) compute count, p95, mean, std
metric_grp = (
    metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(count='count', p95=lambda s: s.quantile(0.95), mean='mean', std='std')
    .reset_index()
    .sort_values('count', ascending=False)
)
metric_summary_top50 = metric_grp.head(50)

# -----------------------
# 2) trace.csv summaries
# -----------------------
trace_cmdbs = pd.Series(trace['cmdb_id'].unique(), name='cmdb_id')
trace_cmdbs_count = trace['cmdb_id'].nunique()

trace_names = pd.Series(trace['trace_name'].unique()[:50], name='trace_name')  # limit to 50
trace_names_count = trace['trace_name'].nunique()

trace_sample = trace.loc[:, ['timestamp', 'cmdb_id', 'trace_name', 'value']].head(20)

# -----------------------
# 3) log.csv summaries
# -----------------------
log_cmdbs = pd.Series(log['cmdb_id'].unique(), name='cmdb_id')
log_cmdbs_count = log['cmdb_id'].nunique()

log_names = pd.Series(log['log_name'].unique()[:50], name='log_name')  # limit to 50
log_names_count = log['log_name'].nunique()

log_sample = log.loc[:, ['timestamp', 'cmdb_id', 'log_name', 'value']].head(20)

# -----------------------
# 4) error_logs.csv summaries
# -----------------------
error_cmdb_counts = (
    error_logs
    .groupby('cmdb_id')
    .size()
    .reset_index(name='error_count')
    .sort_values('error_count', ascending=False)
)
error_cmdb_counts_top20 = error_cmdb_counts.head(20)

error_sample = error_logs.loc[:, ['timestamp', 'cmdb_id', 'message']].head(20)

# Final compact outputs (variables separated by commas so IPython displays them)
metric_cmdbs, metric_cmdbs_count, metric_kpis, metric_kpis_count, metric_summary_top50, trace_cmdbs, trace_cmdbs_count, trace_names, trace_names_count, trace_sample, log_cmdbs, log_cmdbs_count, log_names, log_names_count, log_sample, error_cmdb_counts_top20, error_sample

```
Out[1]:
```
Summary of the loaded telemetry (UTC timestamps):

metric.csv
- Distinct cmdb_id: 12 (adservice, cartservice, checkoutservice, currencyservice, emailservice, frontend, frontend-external, paymentservice, productcatalogservice, recommendationservice, redis, shippingservice).
- Distinct kpi_name: 8 (cpu, diskio, latency-50, latency-90, mem, socket, workload, error).
- Per (cmdb_id, kpi_name) aggregates: a summary table was produced and sorted by number of points (top 50 shown). Most pairs have 25 data points. Example rows: adservice cpu (count=25, p95≈0.7877, mean≈0.7505), adservice mem (count=25, p95≈1.0525e8, mean≈1.0500e8). (Full top-50 table available from the earlier output.)

trace.csv
- Distinct cmdb_id: 8 (checkoutservice, currencyservice, emailservice, frontendservice, paymentservice, productcatalogservice, recommendationservice, root).
- Distinct trace_name: 44 unique trace metrics (list truncated to 50 in output).
- Sample (up to 20 rows): provided — e.g., multiple trace.* entries for checkoutservice at 2024-01-19 10:20:00+00:00 including duration_mean, duration_p95, error_rate, row_count for several from/to relationships.

log.csv
- Distinct cmdb_id: 10 (adservice, cartservice, checkoutservice, currencyservice, emailservice, frontend, paymentservice, recommendationservice, shippingservice, redis).
- Distinct log_name: 2 (log.error_count, log.row_count).
- Sample (up to 20 rows): provided — entries at 2024-01-19 10:20:00+00:00 (and 10:21 for a couple rows). In the sample log.error_count is 0 for all shown rows; log.row_count varies by service.

error_logs.csv
- No error-log rows were found in the loaded file (grouped counts and sample are empty).

If you want any of the summary tables expanded (e.g., full top-50 metric aggregates, more trace or log sample rows, or per-service metric statistics), tell me which table and how many rows to show.

The original code execution output of IPython Kernel is also provided below for reference:

(0                 adservice
1               cartservice
2           checkoutservice
3           currencyservice
4              emailservice
5                  frontend
6         frontend-external
7            paymentservice
8     productcatalogservice
9     recommendationservice
10                    redis
11          shippingservice
Name: cmdb_id, dtype: object, 12, 0           cpu
1        diskio
2    latency-50
3    latency-90
4           mem
5        socket
6      workload
7         error
Name: kpi_name, dtype: object, 8,            cmdb_id    kpi_name  count           p95          mean           std
0        adservice         cpu     25  7.876750e-01  7.505022e-01  5.184797e-02
1        adservice      diskio     25  7.448400e+04  6.260799e+04  8.017267e+03
2        adservice  latency-50     25  3.007846e-03  3.001011e-03  3.542400e-06
3        adservice  latency-90     25  4.615804e-03  4.607999e-03  5.398077e-06
4        adservice         mem     25  1.052534e+08  1.050047e+08  2.223337e+05
..             ...         ...    ...           ...           ...           ...
45  paymentservice      diskio     25  0.000000e+00  0.000000e+00  0.000000e+00
46  paymentservice  latency-50     25  4.443815e-03  3.667400e-03  4.709280e-04
47  paymentservice  latency-90     25  9.956551e-03  7.452083e-03  1.565462e-03
48  paymentservice         mem     25  4.292697e+07  3.808878e+07  3.515333e+06
49  paymentservice      socket     25  3.000000e+00  3.000000e+00  0.000000e+00

[50 rows x 6 columns], 0          checkoutservice
1          currencyservice
2             emailservice
3          frontendservice
4           paymentservice
5    productcatalogservice
6    recommendationservice
7                     root
Name: cmdb_id, dtype: object, 8, 0           trace.from_checkoutservice.duration_mean
1            trace.from_checkoutservice.duration_p95
2              trace.from_checkoutservice.error_rate
3               trace.from_checkoutservice.row_count
4           trace.from_frontendservice.duration_mean
                           ...                      
39          trace.to_recommendationservice.row_count
40    trace.from_recommendationservice.duration_mean
41     trace.from_recommendationservice.duration_p95
42       trace.from_recommendationservice.error_rate
43        trace.from_recommendationservice.row_count
Name: trace_name, Length: 44, dtype: object, 44,                    timestamp          cmdb_id                                trace_name      value
0  2024-01-19 10:20:00+00:00  checkoutservice  trace.from_checkoutservice.duration_mean   0.003735
1  2024-01-19 10:20:00+00:00  checkoutservice   trace.from_checkoutservice.duration_p95   0.005365
2  2024-01-19 10:20:00+00:00  checkoutservice     trace.from_checkoutservice.error_rate   0.000000
3  2024-01-19 10:20:00+00:00  checkoutservice      trace.from_checkoutservice.row_count  88.000000
4  2024-01-19 10:20:00+00:00  checkoutservice  trace.from_frontendservice.duration_mean   0.083468
5  2024-01-19 10:20:00+00:00  checkoutservice   trace.from_frontendservice.duration_p95   0.155467
6  2024-01-19 10:20:00+00:00  checkoutservice     trace.from_frontendservice.error_rate   0.000000
7  2024-01-19 10:20:00+00:00  checkoutservice      trace.from_frontendservice.row_count  11.000000
8  2024-01-19 10:20:00+00:00  checkoutservice             trace.from_root.duration_mean   0.014163
9  2024-01-19 10:20:00+00:00  checkoutservice              trace.from_root.duration_p95   0.063222
10 2024-01-19 10:20:00+00:00  checkoutservice                trace.from_root.error_rate   0.000000
11 2024-01-19 10:20:00+00:00  checkoutservice                 trace.from_root.row_count  42.000000
12 2024-01-19 10:20:00+00:00  checkoutservice    trace.to_checkoutservice.duration_mean   0.003735
13 2024-01-19 10:20:00+00:00  checkoutservice     trace.to_checkoutservice.duration_p95   0.005365
14 2024-01-19 10:20:00+00:00  checkoutservice       trace.to_checkoutservice.error_rate   0.000000
15 2024-01-19 10:20:00+00:00  checkoutservice        trace.to_checkoutservice.row_count  88.000000
16 2024-01-19 10:20:00+00:00  checkoutservice    trace.to_currencyservice.duration_mean   0.000150
17 2024-01-19 10:20:00+00:00  checkoutservice     trace.to_currencyservice.duration_p95   0.000213
18 2024-01-19 10:20:00+00:00  checkoutservice       trace.to_currencyservice.error_rate   0.000000
19 2024-01-19 10:20:00+00:00  checkoutservice        trace.to_currencyservice.row_count  33.000000, 0                adservice
1              cartservice
2          checkoutservice
3          currencyservice
4             emailservice
5                 frontend
6           paymentservice
7    recommendationservice
8          shippingservice
9                    redis
Name: cmdb_id, dtype: object, 10, 0    log.error_count
1      log.row_count
Name: log_name, dtype: object, 2,                    timestamp                cmdb_id         log_name  value
0  2024-01-19 10:20:00+00:00              adservice  log.error_count      0
1  2024-01-19 10:20:00+00:00              adservice    log.row_count    177
2  2024-01-19 10:20:00+00:00            cartservice  log.error_count      0
3  2024-01-19 10:20:00+00:00            cartservice    log.row_count    311
4  2024-01-19 10:20:00+00:00        checkoutservice  log.error_count      0
5  2024-01-19 10:20:00+00:00        checkoutservice    log.row_count     33
6  2024-01-19 10:20:00+00:00        currencyservice  log.error_count      0
7  2024-01-19 10:20:00+00:00        currencyservice    log.row_count    912
8  2024-01-19 10:20:00+00:00           emailservice  log.error_count      0
9  2024-01-19 10:20:00+00:00           emailservice    log.row_count     11
10 2024-01-19 10:20:00+00:00               frontend  log.error_count      0
11 2024-01-19 10:20:00+00:00               frontend    log.row_count   1053
12 2024-01-19 10:20:00+00:00         paymentservice  log.error_count      0
13 2024-01-19 10:20:00+00:00         paymentservice    log.row_count     22
14 2024-01-19 10:20:00+00:00  recommendationservice  log.error_count      0
15 2024-01-19 10:20:00+00:00  recommendationservice    log.row_count    229
16 2024-01-19 10:20:00+00:00        shippingservice  log.error_count      0
17 2024-01-19 10:20:00+00:00        shippingservice    log.row_count    186
18 2024-01-19 10:21:00+00:00              adservice  log.error_count      0
19 2024-01-19 10:21:00+00:00              adservice    log.row_count    443, Empty DataFrame
Columns: [cmdb_id, error_count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# reuse existing 'metric' if present; otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")
# Ensure timestamp parsed to UTC as required
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# 1) Compute global thresholds (before any filtering)
high_kpis = {'cpu', 'diskio', 'latency-50', 'latency-90', 'mem', 'socket', 'error'}
low_kpis = {'workload'}

# compute p95 and p05 per (cmdb_id, kpi_name) on full series
grp = metric.groupby(['cmdb_id', 'kpi_name'])['value']
thresholds = grp.agg(global_p95=lambda s: s.quantile(0.95), global_p05=lambda s: s.quantile(0.05)).reset_index()

# determine orientation and select the global_threshold to use
def orient(row):
    if row['kpi_name'] in low_kpis:
        return 'low'
    else:
        # treat everything else as high-oriented per instructions (only 'workload' is low)
        return 'high'

thresholds['orientation'] = thresholds.apply(orient, axis=1)
thresholds['global_threshold'] = np.where(thresholds['orientation'] == 'high', thresholds['global_p95'], thresholds['global_p05'])

# 2) Filter metric rows inside the incident window (inclusive)
start = pd.to_datetime("2024-01-19 10:17:35", utc=True)
end = pd.to_datetime("2024-01-19 10:47:35", utc=True)
metric_window = metric[(metric['timestamp'] >= start) & (metric['timestamp'] <= end)].copy()

# 3) Merge thresholds into windowed rows and detect anomalies
mw = metric_window.merge(thresholds[['cmdb_id','kpi_name','orientation','global_threshold','global_p95','global_p05']],
                         on=['cmdb_id','kpi_name'], how='left')

# drop rows lacking thresholds (no historical data for that series)
mw = mw.dropna(subset=['global_threshold'])

# determine anomalies per row
def is_anom(row):
    if row['orientation'] == 'high':
        return row['value'] >= row['global_p95']
    else:
        return row['value'] <= row['global_p05']

mw['is_anomaly'] = mw.apply(is_anom, axis=1)
mw['is_high_or_low'] = np.where(mw['orientation']=='high', 'high', 'low')

raw_anomalies = mw[mw['is_anomaly']].copy()
# select requested raw columns
raw_anomalies_small = raw_anomalies.loc[:, ['timestamp','cmdb_id','kpi_name','value','global_threshold','is_high_or_low']].head(20)

# 4) Aggregate anomalies by (cmdb_id, kpi_name)
if not raw_anomalies.empty:
    agg = raw_anomalies.groupby(['cmdb_id','kpi_name','orientation']).agg(
        anomaly_count = ('value','size'),
        earliest_anomaly_timestamp = ('timestamp','min'),
        max_value_in_window = ('value','max'),
        min_value_in_window = ('value','min'),
        global_threshold = ('global_threshold','first')
    ).reset_index()

    # compute severity metric per specification
    def compute_sev(row):
        gt = row['global_threshold']
        if row['orientation'] == 'high':
            # severity = max_value / P95
            if pd.isna(gt) or gt == 0:
                return np.inf if row['max_value_in_window'] > 0 else np.nan
            return row['max_value_in_window'] / gt
        else:
            # low-oriented: severity = P05 / min_value
            minv = row['min_value_in_window']
            if pd.isna(minv) or minv == 0:
                return np.inf if gt > 0 else np.nan
            return gt / minv

    agg['severity_metric'] = agg.apply(compute_sev, axis=1)

    # keep requested columns and sort
    agg_out = agg.loc[:, ['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','global_threshold','severity_metric']]
    agg_out = agg_out.sort_values(['anomaly_count','severity_metric'], ascending=[False, False]).head(20)
else:
    # empty result frames
    agg_out = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','global_threshold','severity_metric'])
    raw_anomalies_small = raw_anomalies_small  # will be empty

# Final compact outputs
agg_out, raw_anomalies_small

```
Out[2]:
```
summary = (
    "Incident window: 2024-01-19 10:17:35 to 2024-01-19 10:47:35 UTC.\n"
    "Global thresholds: per (cmdb_id, kpi_name) P95 used for high-oriented KPIs; P05 used for low-oriented KPI 'workload'.\n\n"
    "Key findings (compact):\n"
    "- Socket anomalies: many services (adservice, cartservice, currencyservice, emailservice, frontend, paymentservice, productcatalogservice, recommendationservice, redis, shippingservice) had 25 anomalies each within the window (i.e., all sampled points met/exceeded their global P95). Max values equal their P95 and severity = 1.0. Earliest anomalies occur at 2024-01-19 10:20:00+00:00.\n"
    "- Disk I/O and some 'error' series show values 0 with global_threshold 0; these were flagged as anomalies by the >=P95 rule but severity is undefined (global_threshold == 0 → NaN severity).\n"
    "- checkoutservice latency-90: 2 anomalies, earliest at 2024-01-19 10:20:00+00:00; max value 0.79925 vs global P95 0.573872 → severity ≈ 1.393 (notable).\n"
    "- emailservice workload (low-oriented): a low anomaly at 2024-01-19 10:20:00+00:00 with value 0.6268 ≤ P05 0.677 → severity ≈ 1.08 (P05 / min_value).\n\n"
    "Notes: many anomalies cluster at 10:20:00+00:00. Some severity metrics are NaN when the global threshold is zero (divide-by-zero). "
    "Recommended next steps: inspect socket and checkoutservice latency traces/logs around 2024-01-19 10:20:00 UTC, and review why diskio/error thresholds are zero (possible data normalization/scale issue)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  global_threshold  severity_metric
5               adservice      socket             25  2024-01-19 10:20:00+00:00              4.00000          4.000000         1.000000
12            cartservice      socket             25  2024-01-19 10:20:00+00:00              4.00000          4.000000         1.000000
25        currencyservice      socket             25  2024-01-19 10:20:00+00:00              3.00000          3.000000         1.000000
32           emailservice      socket             25  2024-01-19 10:20:00+00:00              3.00000          3.000000         1.000000
40               frontend      socket             25  2024-01-19 10:20:00+00:00             23.00000         23.000000         1.000000
49         paymentservice      socket             25  2024-01-19 10:20:00+00:00              3.00000          3.000000         1.000000
56  productcatalogservice      socket             25  2024-01-19 10:20:00+00:00              3.00000          3.000000         1.000000
63  recommendationservice      socket             25  2024-01-19 10:20:00+00:00              4.00000          4.000000         1.000000
68                  redis      socket             25  2024-01-19 10:20:00+00:00              4.00000          4.000000         1.000000
73        shippingservice      socket             25  2024-01-19 10:20:00+00:00              2.00000          2.000000         1.000000
8             cartservice      diskio             25  2024-01-19 10:20:00+00:00              0.00000          0.000000              NaN
21        currencyservice      diskio             25  2024-01-19 10:20:00+00:00              0.00000          0.000000              NaN
28           emailservice      diskio             25  2024-01-19 10:20:00+00:00              0.00000          0.000000              NaN
35               frontend      diskio             25  2024-01-19 10:20:00+00:00              0.00000          0.000000              NaN
36               frontend       error             25  2024-01-19 10:20:00+00:00              0.00000          0.000000              NaN
42      frontend-external       error             25  2024-01-19 10:20:00+00:00              0.00000          0.000000              NaN
45         paymentservice      diskio             25  2024-01-19 10:20:00+00:00              0.00000          0.000000              NaN
52  productcatalogservice      diskio             25  2024-01-19 10:20:00+00:00              0.00000          0.000000              NaN
59  recommendationservice      diskio             25  2024-01-19 10:20:00+00:00              0.00000          0.000000              NaN
16        checkoutservice  latency-90              2  2024-01-19 10:20:00+00:00              0.79925          0.573872         1.392732,                    timestamp            cmdb_id    kpi_name      value  global_threshold is_high_or_low
0  2024-01-19 10:20:00+00:00          adservice         cpu   0.789184          0.787675           high
5  2024-01-19 10:20:00+00:00          adservice      socket   4.000000          4.000000           high
8  2024-01-19 10:20:00+00:00        cartservice      diskio   0.000000          0.000000           high
9  2024-01-19 10:20:00+00:00        cartservice  latency-50   0.003169          0.003152           high
10 2024-01-19 10:20:00+00:00        cartservice  latency-90   0.004904          0.004873           high
12 2024-01-19 10:20:00+00:00        cartservice      socket   4.000000          4.000000           high
16 2024-01-19 10:20:00+00:00    checkoutservice  latency-90   0.799250          0.573872           high
21 2024-01-19 10:20:00+00:00    currencyservice      diskio   0.000000          0.000000           high
25 2024-01-19 10:20:00+00:00    currencyservice      socket   3.000000          3.000000           high
28 2024-01-19 10:20:00+00:00       emailservice      diskio   0.000000          0.000000           high
29 2024-01-19 10:20:00+00:00       emailservice  latency-50   0.003214          0.003075           high
30 2024-01-19 10:20:00+00:00       emailservice  latency-90   0.005070          0.004734           high
32 2024-01-19 10:20:00+00:00       emailservice      socket   3.000000          3.000000           high
33 2024-01-19 10:20:00+00:00       emailservice    workload   0.626800          0.677000            low
35 2024-01-19 10:20:00+00:00           frontend      diskio   0.000000          0.000000           high
36 2024-01-19 10:20:00+00:00           frontend       error   0.000000          0.000000           high
40 2024-01-19 10:20:00+00:00           frontend      socket  23.000000         23.000000           high
42 2024-01-19 10:20:00+00:00  frontend-external       error   0.000000          0.000000           high
45 2024-01-19 10:20:00+00:00     paymentservice      diskio   0.000000          0.000000           high
49 2024-01-19 10:20:00+00:00     paymentservice      socket   3.000000          3.000000           high)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# reuse existing 'trace' if present, otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")

# Ensure timestamp parsed to UTC as required
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# 1) Compute global P95 and P05 per (cmdb_id, trace_name) using full data
trace_thresh = trace.groupby(['cmdb_id','trace_name'])['value'].agg(
    global_p95 = lambda s: s.quantile(0.95),
    global_p05 = lambda s: s.quantile(0.05)
).reset_index()

# determine orientation per instruction
def trace_orientation(name):
    n = name.lower()
    if 'row_count' in n:
        return 'low'
    if ('duration' in n) or ('p95' in n) or ('error_rate' in n):
        return 'high'
    return 'other'

trace_thresh['orientation'] = trace_thresh['trace_name'].apply(trace_orientation)
# choose applicable global_threshold for convenience (use p95 for high, p05 for low)
trace_thresh['global_threshold'] = np.where(trace_thresh['orientation']=='high',
                                           trace_thresh['global_p95'],
                                           trace_thresh['global_p05'])

# 2) Filter trace rows to the incident window (inclusive)
start = pd.to_datetime("2024-01-19 10:17:35", utc=True)
end = pd.to_datetime("2024-01-19 10:47:35", utc=True)
trace_window = trace[(trace['timestamp'] >= start) & (trace['timestamp'] <= end)].copy()

# 3) Detect anomalies by merging thresholds and applying rules
tw = trace_window.merge(trace_thresh[['cmdb_id','trace_name','orientation','global_p95','global_p05','global_threshold']],
                        on=['cmdb_id','trace_name'], how='left')

# drop rows without thresholds (orientation == 'other' or missing)
tw = tw.dropna(subset=['orientation','global_threshold'])

# anomaly detection per row
def is_trace_anom(row):
    if row['orientation'] == 'high':
        # anomaly if value >= global_p95
        return row['value'] >= row['global_p95']
    else:
        # low: anomaly if value <= global_p05
        return row['value'] <= row['global_p05']

tw['is_anomaly'] = tw.apply(is_trace_anom, axis=1)
tw['is_high_or_low'] = np.where(tw['orientation']=='high', 'high', 'low')

anoms = tw[tw['is_anomaly']].copy()

# compute per-row severity for aggregation:
# for high: value / global_p95
# for low: global_p05 / value (careful with zero)
def row_severity(row):
    if row['orientation'] == 'high':
        gp = row['global_p95']
        return np.nan if gp == 0 or pd.isna(gp) else row['value'] / gp
    else:
        gp = row['global_p05']
        val = row['value']
        return np.nan if val == 0 or pd.isna(val) else gp / val

anoms['severity'] = anoms.apply(row_severity, axis=1)

# 4) Aggregate by cmdb_id (limit 20 rows)
if not anoms.empty:
    # anomaly count and earliest timestamp per cmdb_id
    agg_cm = anoms.groupby('cmdb_id').agg(
        anomaly_count = ('value','size'),
        earliest_anomaly_timestamp = ('timestamp','min'),
        max_severity = ('severity','max')
    ).reset_index()

    # top 3 trace_names contributing to anomalies per cmdb_id
    top_traces = (anoms.groupby(['cmdb_id','trace_name']).size().reset_index(name='count')
                  .sort_values(['cmdb_id','count'], ascending=[True, False]))
    # collect top 3 per cmdb_id
    top3_list = top_traces.groupby('cmdb_id').head(3).groupby('cmdb_id')['trace_name'].agg(lambda s: ';'.join(s)).reset_index()
    agg_cm = agg_cm.merge(top3_list, on='cmdb_id', how='left')
    agg_cm = agg_cm.rename(columns={'trace_name':'top_3_trace_names'})
    agg_cm = agg_cm.sort_values(['anomaly_count','max_severity'], ascending=[False, False]).head(20)
else:
    agg_cm = pd.DataFrame(columns=['cmdb_id','anomaly_count','earliest_anomaly_timestamp','max_severity','top_3_trace_names'])

# 5) Up to 20 raw anomaly rows for inspection
raw_anoms_small = anoms.loc[:, ['timestamp','cmdb_id','trace_name','value','global_threshold','is_high_or_low']].head(20)

# Final compact outputs
agg_cm, raw_anoms_small

```
Out[3]:
```
Incident window: 2024-01-19 10:17:35 — 2024-01-19 10:47:35 UTC.

Key findings (compact):
- The anomaly activity is concentrated at 2024-01-19 10:20:00+00:00.
- Top affected services (by anomaly count):
  - checkoutservice — 248 anomalies, earliest 10:20:00, max severity ≈ 2.04. Top contributing traces include trace.from_checkoutservice.error_rate, trace.from_checkoutservice.row_count, trace.from_frontendservice.*.
  - root — 217 anomalies, earliest 10:20:00, max severity ≈ 2.60. Top contributing traces include trace.to_checkoutservice.error_rate and related traces.
  - frontendservice — 217 anomalies, earliest 10:20:00, max severity ≈ 1.69.
  - recommendationservice — 156 anomalies, earliest 10:20:00, max severity = 1.6.
  - productcatalogservice — 125 anomalies, earliest 10:20:00, max severity ≈ 1.87.
  - currencyservice — 93 anomalies, earliest 10:20:00, max severity = 2.2.
  - paymentservice, emailservice — 62 anomalies each.
- Many anomalies are trace error_rate/duration types and row_count drops:
  - Numerous error_rate/duration series had global P95 = 0, so rows with value == 0 were flagged (global_threshold shown as 0). This can produce misleading flags (and NaN/undefined severities) — inspect threshold computation and baseline zeros.
  - Several services show row_count anomalies where row_count in-window is significantly below the global P05 (e.g., checkoutservice row_count 88 <= P05 127.4), indicating traffic/throughput drops.
- Notable severities (value / global_P95 or P05 / min_value): root and paymentservice ≈ 2.60, currencyservice 2.2, checkoutservice ≈ 2.04 — these indicate values roughly 2× their usual 95th-percentile (or P05 breaches for row_count).

Raw example anomalies (few inspected):
- checkoutservice: many trace.*.error_rate rows with global_threshold 0 (value 0 flagged), and trace.*.row_count rows where counts (e.g., 88, 42, 11, 33, 22) are below their global P05.
- currencyservice and frontend-related traces show similar patterns (error_rate zeros + row_count drops).

Immediate recommendations:
1. Investigate services around checkoutservice (and calls to/from checkoutservice) at ~10:20 UTC — this is the focal point.
2. Verify why many trace error_rate/duration global_P95 values are zero (data normalization/baseline issues) because that makes the anomaly rule noisy.
3. Inspect trace row_count decreases (traffic/throughput drops) for checkoutservice, frontendservice, currencyservice, productcatalogservice and paymentservice.

If you want, I can:
- show the full top trace_names per affected service,
- expand raw anomaly rows for a specific cmdb_id,
- or recompute thresholds ignoring zero baselines to reduce false positives.

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id  anomaly_count earliest_anomaly_timestamp  max_severity                                  top_3_trace_names
0        checkoutservice            248  2024-01-19 10:20:00+00:00      2.040235  trace.from_checkoutservice.error_rate;trace.fr...
7                   root            217  2024-01-19 10:20:00+00:00      2.596008  trace.to_checkoutservice.error_rate;trace.to_c...
3        frontendservice            217  2024-01-19 10:20:00+00:00      1.687808  trace.from_frontendservice.error_rate;trace.fr...
6  recommendationservice            156  2024-01-19 10:20:00+00:00      1.600000  trace.from_frontendservice.error_rate;trace.fr...
5  productcatalogservice            125  2024-01-19 10:20:00+00:00      1.866667  trace.from_checkoutservice.error_rate;trace.fr...
1        currencyservice             93  2024-01-19 10:20:00+00:00      2.200000  trace.from_checkoutservice.error_rate;trace.fr...
4         paymentservice             62  2024-01-19 10:20:00+00:00      2.596008  trace.from_checkoutservice.error_rate;trace.fr...
2           emailservice             62  2024-01-19 10:20:00+00:00      1.600000  trace.from_checkoutservice.error_rate;trace.fr...,                    timestamp          cmdb_id                                 trace_name  value  global_threshold is_high_or_low
2  2024-01-19 10:20:00+00:00  checkoutservice      trace.from_checkoutservice.error_rate    0.0               0.0           high
3  2024-01-19 10:20:00+00:00  checkoutservice       trace.from_checkoutservice.row_count   88.0             127.4            low
6  2024-01-19 10:20:00+00:00  checkoutservice      trace.from_frontendservice.error_rate    0.0               0.0           high
7  2024-01-19 10:20:00+00:00  checkoutservice       trace.from_frontendservice.row_count   11.0              16.2            low
10 2024-01-19 10:20:00+00:00  checkoutservice                 trace.from_root.error_rate    0.0               0.0           high
11 2024-01-19 10:20:00+00:00  checkoutservice                  trace.from_root.row_count   42.0              68.8            low
14 2024-01-19 10:20:00+00:00  checkoutservice        trace.to_checkoutservice.error_rate    0.0               0.0           high
15 2024-01-19 10:20:00+00:00  checkoutservice         trace.to_checkoutservice.row_count   88.0             127.4            low
18 2024-01-19 10:20:00+00:00  checkoutservice        trace.to_currencyservice.error_rate    0.0               0.0           high
19 2024-01-19 10:20:00+00:00  checkoutservice         trace.to_currencyservice.row_count   33.0              52.0            low
22 2024-01-19 10:20:00+00:00  checkoutservice           trace.to_emailservice.error_rate    0.0               0.0           high
23 2024-01-19 10:20:00+00:00  checkoutservice            trace.to_emailservice.row_count   11.0              14.6            low
26 2024-01-19 10:20:00+00:00  checkoutservice         trace.to_paymentservice.error_rate    0.0               0.0           high
27 2024-01-19 10:20:00+00:00  checkoutservice          trace.to_paymentservice.row_count   11.0              15.4            low
30 2024-01-19 10:20:00+00:00  checkoutservice  trace.to_productcatalogservice.error_rate    0.0               0.0           high
31 2024-01-19 10:20:00+00:00  checkoutservice   trace.to_productcatalogservice.row_count   22.0              36.6            low
34 2024-01-19 10:20:00+00:00  currencyservice      trace.from_checkoutservice.error_rate    0.0               0.0           high
35 2024-01-19 10:20:00+00:00  currencyservice       trace.from_checkoutservice.row_count   33.0              52.0            low
38 2024-01-19 10:20:00+00:00  currencyservice      trace.from_frontendservice.error_rate    0.0               0.0           high
39 2024-01-19 10:20:00+00:00  currencyservice       trace.from_frontendservice.row_count  878.0            1376.2            low)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Candidates
candidates = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice', 'paymentservice']

# Load or reuse metric and trace DataFrames, ensure timestamps are parsed to UTC
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# Incident window (UTC, inclusive)
start = pd.to_datetime("2024-01-19 10:17:35", utc=True)
end = pd.to_datetime("2024-01-19 10:47:35", utc=True)

# -------------------------
# METRIC: global thresholds
# -------------------------
# Compute global P95 and P05 per (cmdb_id, kpi_name) using full series
metric_thresh = metric.groupby(['cmdb_id','kpi_name'])['value'].agg(
    global_p95 = lambda s: s.quantile(0.95),
    global_p05 = lambda s: s.quantile(0.05)
).reset_index()

# Orientation: 'workload' is low, others are high
metric_thresh['orientation'] = np.where(metric_thresh['kpi_name']=='workload','low','high')
metric_thresh['global_threshold'] = np.where(metric_thresh['orientation']=='high', metric_thresh['global_p95'], metric_thresh['global_p05'])

# Filter metric rows to window and to candidate components
metric_window = metric[(metric['timestamp'] >= start) & (metric['timestamp'] <= end) & (metric['cmdb_id'].isin(candidates))].copy()

# Merge thresholds
mw = metric_window.merge(metric_thresh[['cmdb_id','kpi_name','orientation','global_p95','global_p05','global_threshold']],
                         on=['cmdb_id','kpi_name'], how='left')

# Drop rows without thresholds (no historical data)
mw = mw.dropna(subset=['orientation','global_threshold'])

# Detect metric anomalies
def metric_is_anom(row):
    if row['orientation'] == 'high':
        return row['value'] >= row['global_p95']
    else:
        return row['value'] <= row['global_p05']

mw['is_anomaly'] = mw.apply(metric_is_anom, axis=1)
mw['is_high_or_low'] = np.where(mw['orientation']=='high','high','low')

# severity per metric anomaly row
def metric_severity(row):
    if row['orientation']=='high':
        gp = row['global_p95']
        return np.nan if pd.isna(gp) or gp == 0 else row['value'] / gp
    else:
        gp = row['global_p05']
        val = row['value']
        return np.nan if pd.isna(val) or val == 0 else gp / val

mw['severity'] = mw.apply(metric_severity, axis=1)

metric_anoms = mw[mw['is_anomaly']].copy()

# Aggregate metric anomalies per candidate
metric_agg = (metric_anoms.groupby('cmdb_id').agg(
    metric_anomaly_count = ('value','size'),
    metric_earliest_anomaly_timestamp = ('timestamp','min'),
    metric_max_severity = ('severity','max')
).reindex(candidates).reset_index().rename(columns={'index':'cmdb_id'}))

# Ensure all candidates present (fill zeros/NA for missing)
metric_agg = metric_agg.rename(columns={'cmdb_id':'component'})
metric_agg['metric_anomaly_count'] = metric_agg['metric_anomaly_count'].fillna(0).astype(int)
metric_agg['metric_earliest_anomaly_timestamp'] = pd.to_datetime(metric_agg['metric_earliest_anomaly_timestamp'])
metric_agg['metric_earliest_anomaly_timestamp'] = metric_agg['metric_earliest_anomaly_timestamp'].where(~metric_agg['metric_earliest_anomaly_timestamp'].isna(), pd.NaT)
metric_agg['metric_max_severity'] = metric_agg['metric_max_severity'].astype(float)

# -------------------------
# TRACE: global thresholds
# -------------------------
# Compute global p95 and p05 per (cmdb_id, trace_name)
trace_thresh = trace.groupby(['cmdb_id','trace_name'])['value'].agg(
    global_p95 = lambda s: s.quantile(0.95),
    global_p05 = lambda s: s.quantile(0.05)
).reset_index()

# Determine orientation per trace_name
def trace_orient(name):
    n = name.lower()
    if 'row_count' in n:
        return 'low'
    if ('duration' in n) or ('p95' in n) or ('error_rate' in n):
        return 'high'
    return 'other'

trace_thresh['orientation'] = trace_thresh['trace_name'].apply(trace_orient)
trace_thresh['global_threshold'] = np.where(trace_thresh['orientation']=='high', trace_thresh['global_p95'], trace_thresh['global_p05'])

# Filter trace rows to window and candidates
trace_window = trace[(trace['timestamp'] >= start) & (trace['timestamp'] <= end) & (trace['cmdb_id'].isin(candidates))].copy()

# Merge thresholds
tw = trace_window.merge(trace_thresh[['cmdb_id','trace_name','orientation','global_p95','global_p05','global_threshold']],
                        on=['cmdb_id','trace_name'], how='left')

# Keep only high/low oriented traces with thresholds
tw = tw.dropna(subset=['orientation','global_threshold'])

# Detect trace anomalies
def trace_is_anom(row):
    if row['orientation']=='high':
        return row['value'] >= row['global_p95']
    else:
        return row['value'] <= row['global_p05']

tw['is_anomaly'] = tw.apply(trace_is_anom, axis=1)
tw['is_high_or_low'] = np.where(tw['orientation']=='high','high','low')

# severity per trace anomaly row
def trace_severity(row):
    if row['orientation']=='high':
        gp = row['global_p95']
        return np.nan if pd.isna(gp) or gp == 0 else row['value'] / gp
    else:
        gp = row['global_p05']
        val = row['value']
        return np.nan if pd.isna(val) or val == 0 else gp / val

tw['severity'] = tw.apply(trace_severity, axis=1)

trace_anoms = tw[tw['is_anomaly']].copy()

# Aggregate trace anomalies per candidate
trace_agg = (trace_anoms.groupby('cmdb_id').agg(
    trace_anomaly_count = ('value','size'),
    trace_earliest_anomaly_timestamp = ('timestamp','min'),
    trace_max_severity = ('severity','max')
).reindex(candidates).reset_index().rename(columns={'index':'cmdb_id'}))

trace_agg = trace_agg.rename(columns={'cmdb_id':'component'})
trace_agg['trace_anomaly_count'] = trace_agg['trace_anomaly_count'].fillna(0).astype(int)
trace_agg['trace_earliest_anomaly_timestamp'] = pd.to_datetime(trace_agg['trace_earliest_anomaly_timestamp'])
trace_agg['trace_earliest_anomaly_timestamp'] = trace_agg['trace_earliest_anomaly_timestamp'].where(~trace_agg['trace_earliest_anomaly_timestamp'].isna(), pd.NaT)
trace_agg['trace_max_severity'] = trace_agg['trace_max_severity'].astype(float)

# -------------------------
# Combine metric and trace aggregates
# -------------------------
agg = metric_agg.merge(trace_agg, on='component', how='outer')[[
    'component',
    'metric_anomaly_count','metric_earliest_anomaly_timestamp','metric_max_severity',
    'trace_anomaly_count','trace_earliest_anomaly_timestamp','trace_max_severity'
]]

# Replace NaNs for counts already handled; ensure severity columns numeric
agg['metric_max_severity'] = agg['metric_max_severity'].replace({np.nan: np.nan})
agg['trace_max_severity'] = agg['trace_max_severity'].replace({np.nan: np.nan})

# Sort by trace_max_severity desc then trace_anomaly_count desc
agg_sort = agg.copy()
# for sorting, treat NaN severities as 0
agg_sort['trace_max_severity_sort'] = agg_sort['trace_max_severity'].fillna(0)
agg_sort = agg_sort.sort_values(['trace_max_severity_sort','trace_anomaly_count'], ascending=[False, False]).drop(columns=['trace_max_severity_sort'])

# Limit to candidate components (should be 6 rows)
agg_result = agg_sort.reset_index(drop=True)

# -------------------------
# Raw trace anomalies for top component by trace_max_severity
# -------------------------
# Identify top component (highest trace_max_severity, tie-broken by trace_anomaly_count)
if not agg_result.empty:
    top_comp = agg_result.iloc[0]['component']
    raw_trace_anoms_small = (trace_anoms[trace_anoms['cmdb_id']==top_comp]
                             .loc[:, ['timestamp','cmdb_id','trace_name','value','global_threshold','is_high_or_low']]
                             .sort_values('timestamp')
                             .head(10))
else:
    raw_trace_anoms_small = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','global_threshold','is_high_or_low'])

# Final compact outputs
agg_result, raw_trace_anoms_small

```
Out[4]:
```
Summary of aggregated anomaly results for the six candidate services (incident window 2024-01-19 10:17:35 — 10:47:35 UTC). Rows are ordered by trace_max_severity (desc):

- paymentservice
  - metric_anomaly_count: 60
  - metric_earliest_anomaly_timestamp: 2024-01-19 10:20:00+00:00
  - metric_max_severity: 1.194961
  - trace_anomaly_count: 62
  - trace_earliest_anomaly_timestamp: 2024-01-19 10:20:00+00:00
  - trace_max_severity: 2.596008

- currencyservice
  - metric_anomaly_count: 60
  - metric_earliest_anomaly_timestamp: 2024-01-19 10:20:00+00:00
  - metric_max_severity: 1.015144
  - trace_anomaly_count: 93
  - trace_earliest_anomaly_timestamp: 2024-01-19 10:20:00+00:00
  - trace_max_severity: 2.200000

- checkoutservice
  - metric_anomaly_count: 12
  - metric_earliest_anomaly_timestamp: 2024-01-19 10:20:00+00:00
  - metric_max_severity: 1.392732
  - trace_anomaly_count: 248
  - trace_earliest_anomaly_timestamp: 2024-01-19 10:20:00+00:00
  - trace_max_severity: 2.040235

- productcatalogservice
  - metric_anomaly_count: 60
  - metric_earliest_anomaly_timestamp: 2024-01-19 10:20:00+00:00
  - metric_max_severity: 1.024650
  - trace_anomaly_count: 125
  - trace_earliest_anomaly_timestamp: 2024-01-19 10:20:00+00:00
  - trace_max_severity: 1.866667

- recommendationservice
  - metric_anomaly_count: 60
  - metric_earliest_anomaly_timestamp: 2024-01-19 10:20:00+00:00
  - metric_max_severity: 1.020699
  - trace_anomaly_count: 156
  - trace_earliest_anomaly_timestamp: 2024-01-19 10:20:00+00:00
  - trace_max_severity: 1.600000

- emailservice
  - metric_anomaly_count: 60
  - metric_earliest_anomaly_timestamp: 2024-01-19 10:20:00+00:00
  - metric_max_severity: 1.080089
  - trace_anomaly_count: 62
  - trace_earliest_anomaly_timestamp: 2024-01-19 10:20:00+00:00
  - trace_max_severity: 1.600000

Top component by trace_max_severity: paymentservice. Example raw trace-anomaly rows for paymentservice (few inspected):
- 2024-01-19 10:20:00+00:00 — trace.from_checkoutservice.error_rate — value 0.0 — global_threshold 0.0 — high
- 2024-01-19 10:20:00+00:00 — trace.from_checkoutservice.row_count — value 11.0 — global_threshold 15.4 — low
- 2024-01-19 10:20:00+00:00 — trace.from_root.row_count — value 4.0 — global_threshold 8.8 — low
- Additional repeated error_rate rows at 10:21, 10:22, 10:23 for checkout→payment and root→payment.

Interpretation and immediate notes:
- Most anomalies cluster at ~10:20:00 UTC.
- Many trace error_rate global thresholds are zero, causing zero-valued error_rate rows to be flagged (these zero-threshold cases can produce misleading flags and undefined severities).
- There are clear row_count drops for paymentservice (and other services) vs their global P05 (e.g., row_count 11 <= P05 15.4; row_count 4 <= P05 8.8), indicating throughput drops.
- Recommended next steps: investigate paymentservice interactions with checkoutservice around 10:20 UTC (trace flows and logs), and verify why many trace error_rate baselines are zero to reduce false positives.

The original code execution output of IPython Kernel is also provided below for reference:

(               component  metric_anomaly_count metric_earliest_anomaly_timestamp  metric_max_severity  trace_anomaly_count trace_earliest_anomaly_timestamp  trace_max_severity
0         paymentservice                    60         2024-01-19 10:20:00+00:00             1.194961                   62        2024-01-19 10:20:00+00:00            2.596008
1        currencyservice                    60         2024-01-19 10:20:00+00:00             1.015144                   93        2024-01-19 10:20:00+00:00            2.200000
2        checkoutservice                    12         2024-01-19 10:20:00+00:00             1.392732                  248        2024-01-19 10:20:00+00:00            2.040235
3  productcatalogservice                    60         2024-01-19 10:20:00+00:00             1.024650                  125        2024-01-19 10:20:00+00:00            1.866667
4  recommendationservice                    60         2024-01-19 10:20:00+00:00             1.020699                  156        2024-01-19 10:20:00+00:00            1.600000
5           emailservice                    60         2024-01-19 10:20:00+00:00             1.080089                   62        2024-01-19 10:20:00+00:00            1.600000,                     timestamp         cmdb_id                             trace_name  value  global_threshold is_high_or_low
54  2024-01-19 10:20:00+00:00  paymentservice  trace.from_checkoutservice.error_rate    0.0               0.0           high
55  2024-01-19 10:20:00+00:00  paymentservice   trace.from_checkoutservice.row_count   11.0              15.4            low
58  2024-01-19 10:20:00+00:00  paymentservice             trace.from_root.error_rate    0.0               0.0           high
59  2024-01-19 10:20:00+00:00  paymentservice              trace.from_root.row_count    4.0               8.8            low
150 2024-01-19 10:21:00+00:00  paymentservice  trace.from_checkoutservice.error_rate    0.0               0.0           high
154 2024-01-19 10:21:00+00:00  paymentservice             trace.from_root.error_rate    0.0               0.0           high
246 2024-01-19 10:22:00+00:00  paymentservice  trace.from_checkoutservice.error_rate    0.0               0.0           high
250 2024-01-19 10:22:00+00:00  paymentservice             trace.from_root.error_rate    0.0               0.0           high
342 2024-01-19 10:23:00+00:00  paymentservice  trace.from_checkoutservice.error_rate    0.0               0.0           high
346 2024-01-19 10:23:00+00:00  paymentservice             trace.from_root.error_rate    0.0               0.0           high)```
```